## **BactHeCom**
### Preprocessing
This notebook pretends to document the process of loading tables from SQLite database and preprocess, recode and merge them for further modeling.



### Envirnonment setup

In [1]:
# load libraries
import os
from typing import Optional, List
import sqlite3
import pandas as pd
import numpy as np
import requests
from datetime import datetime
from dateutil.relativedelta import relativedelta
from pandas_profiling import ProfileReport
from utils import *

import warnings
warnings.filterwarnings('ignore')

/tmp/ipykernel_2475318/74020452.py:10: DeprecationWarning: `import pandas_profiling` is going to be deprecated by April 1st. Please use `import ydata_profiling` instead.
  from pandas_profiling import ProfileReport


### Paths 

In [2]:
# paths
DATA_DIR = "../data"
BACTAHECOM_DB_PATH=f"{DATA_DIR}/db_bacthecom_v6.db"

RESULTS_DIR = "../results"
os.makedirs(RESULTS_DIR, exist_ok=True)

### Load DB

In [3]:
def load_sqlite_tables(
    db_path: str, drop_tables: Optional[List[str]] = None) -> dict:
    """Load tables from SQLite database and return a dictionary of DataFrames."""
    
    if not os.path.exists(db_path):
        raise FileNotFoundError(
            f"Database path {db_path} does not exist. Please check the path and try again."
        )

    dfs = {}

    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = [table[0] for table in cursor.fetchall()]

        # Drop tables if requested
        if drop_tables:
            tables = [t for t in tables if t not in drop_tables]

        for table in tables:
            print(f"Loading table: {table}")
            dfs[table] = pd.read_sql_query(f"SELECT * FROM {table}", conn)

            # format fecha columns into datetime
            fecha_cols = [col for col in dfs[table].columns if 'fecha' in col.lower()]
            for col in fecha_cols:
                dfs[table][col] = pd.to_datetime(dfs[table][col], errors='coerce').dt.date
            
            # summary
            print(f"Shape of {table}: {dfs[table].shape}")
            print(f"Features in {table}: {dfs[table].columns.tolist()}")
            print("===========================================")
            

    return dfs


In [4]:
# Load sqlite3 DB
dfs = load_sqlite_tables(BACTAHECOM_DB_PATH, drop_tables=['semantic_mapping', 'sqlite_sequence'])

Loading table: paciente
Shape of paciente: (3049, 3)
Features in paciente: ['record_id', 'sexo', 'fecha_nacimiento']
Loading table: antibiograma
Shape of antibiograma: (151653, 4)
Features in antibiograma: ['episode_id', 'antimicrobiano', 'cmi', 'interpretacion']
Loading table: episodio_infeccion
Shape of episodio_infeccion: (7287, 9)
Features in episodio_infeccion: ['episode_id', 'record_id', 'fecha_ingreso', 'fecha_cultivo', 'area_hosp', 'id_cultivo', 'especimen', 'microorganismo', 'fenotipo_resistencia']
Loading table: comorbilidad
Shape of comorbilidad: (3412, 34)
Features in comorbilidad: ['record_id', 'fecha_ingreso', 'infarto', 'insuficiencia_cardiaca', 'evp', 'e_cerebrovascular', 'demencia', 'e_pulmonar_cronica', 'ulcera_peptica', 'colagenopatia', 'hemiplejia', 'erc', 'neoplasia_tratamiento_activo', 'neoplasia_solida_metastasica', 'neoplasia_solida_no_metastasica', 'tipo_cancer', 'linfoma', 'leucemia', 'sida', 'puntaje_child_pugh', 'hepatopatia_ligera', 'hepatopatia_moderada_o_

### Harmonization and merging approach
1) Recode features to one-hot encoding or categorical as needed.
2) Uniformize date and feature names across tables.
3) Drop irrelevente features for the model
4) Merge tables based on `record_id` and `fecha_ingreso` (when the positive hemoculture was taken for that episode) to create a final dataset for modeling.
5) Study % missingness in variables and apply imputation strategies if needed.

## **tbl_pacientes**

In [5]:
tbl_paciente = dfs['paciente'].copy()
print(f"Shape of paciente table: {tbl_paciente.shape}")
tbl_paciente.head()

Shape of paciente table: (3049, 3)


,record_id,sexo,fecha_nacimiento
0,1,Hombre,1943-03-28
1,2,Mujer,1959-12-16
2,3,Hombre,1976-11-14
3,4,Hombre,1992-09-27
4,5,Mujer,1994-07-13


There are **3049** patient entries, but some patients may have multiple episodes of bacteremia

CHANGES
- Recode `sexo` to 0/1

In [6]:
# Recode sexo to 0/1
tbl_paciente['sexo'] = tbl_paciente['sexo'].map({'Hombre': 0, 'Mujer': 1})

## **tbl_factores_riesgo_infeccion_bmr**

In [7]:
tbl_factores_bmr = dfs['factores_riesgo_infeccion_bmr'].copy()
print(f"Shape of factores_riesgo_infeccion_bmr table: {tbl_factores_bmr.shape}")
tbl_factores_bmr.head()

Shape of factores_riesgo_infeccion_bmr table: (3413, 16)


,record_id,fecha_ingreso,hospit_ano_previo,hospit_mes_previo,hospit_ano_previo_uci,cirugia_previa_sin_implante,cirugia_previa_con_implante,asistencia_sanitaria_prev,hemodialisis_permanente,dialisis_peritoneal,cateter_venoso,sonda_urinaria,sonda_nasogastrica,derivacion_ventriculoper,valvula_prot_cardiaca,portador_otros_disposit
0,1,2021-08-24,0,0,0,0.0,1.0,1.0,0,0,0,0,1,0,0,0
1,2,2023-06-18,0,0,0,1.0,1.0,0.0,0,0,0,0,1,0,0,0
2,3,2022-02-11,1,0,1,NaN,NaN,NaN,0,0,0,0,1,0,0,0
3,4,2021-05-15,0,0,0,0.0,0.0,0.0,0,0,0,0,1,0,0,0
4,5,2021-07-04,0,0,0,1.0,0.0,1.0,0,0,0,0,0,0,0,0


## **tbl_episodios**

In [8]:
tbl_episodios = dfs['episodio_ingreso'].copy()
print(f"Shape of episodio_ingreso table: {tbl_episodios.shape}")
tbl_episodios.head()

Shape of episodio_ingreso table: (3610, 17)


,record_id,fecha_ingreso,fecha_alta,organo_aparato,foco_controlable,IRAs_nosocomial,mortalidad,fecha_mortalidad,dias_hemocultivo_mortalidad,mortalidad_30_dias,uci_por_el_episodio,duracion_UCI,dias_hemocultivo_ingresoUCI,en_uci_antes_del_hemocultivo,mujer_gestante,codigo_postal,paciente_residencia
0,1,2021-08-24,2021-09-09,None,NaN,Si,1,2021-09-09,9.0,1,1,14,1.0,1.0,0,4007,0.0
1,2,2023-06-18,2023-07-13,None,NaN,Si,0,NaT,NaN,0,1,3,NaN,NaN,0,4740,0.0
2,3,2022-02-11,2022-04-06,Infeccion de cateter vascular,1.0,Si,0,NaT,NaN,0,1,24,17.0,1.0,0,4700,NaN
3,4,2021-05-15,2021-07-28,None,NaN,Si,0,NaT,NaN,0,1,42,NaN,NaN,0,18800,0.0
4,5,2021-07-04,2022-01-04,Infeccion tracto respiratorio inferior,0.0,Si,0,NaT,NaN,0,1,78,NaN,NaN,0,23009,0.0


OBSERVATIONS

- Check why there are 3610 entries? Duplicated events?

We should have 3412 entries (one per episode of `record_id`+`fecha_ingreso`)


In [9]:
# get duplicates of record_id + fecha_ingreso
dups = tbl_episodios[tbl_episodios.duplicated(subset=['record_id', 'fecha_ingreso'],keep=False)]
dups

,record_id,fecha_ingreso,fecha_alta,organo_aparato,foco_controlable,IRAs_nosocomial,mortalidad,fecha_mortalidad,dias_hemocultivo_mortalidad,mortalidad_30_dias,uci_por_el_episodio,duracion_UCI,dias_hemocultivo_ingresoUCI,en_uci_antes_del_hemocultivo,mujer_gestante,codigo_postal,paciente_residencia
26,27,2020-06-12,2020-07-26,Fiebre sin foco,0.0,Si,1,2020-07-26,30.0,1,0,0,NaN,NaN,0,11500,0.0
27,27,2020-06-12,2020-07-26,Infección de la vía urinaria superior,1.0,Si,1,2020-07-26,18.0,1,0,0,NaN,NaN,0,11500,0.0
28,27,2020-06-12,2020-07-26,Fiebre sin foco,0.0,Si,1,2020-07-26,23.0,1,0,0,NaN,NaN,0,11500,0.0
36,34,2021-08-08,2022-03-04,Infeccion tracto respiratorio inferior,0.0,Si,0,NaT,NaN,0,1,188,17.0,1.0,0,11205,0.0
37,34,2021-08-08,2022-03-04,Infeccion tracto respiratorio inferior,0.0,Si,0,NaT,NaN,0,1,188,NaN,NaN,0,11205,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3585,3028,2022-10-05,2022-12-07,Infeccion de cateter vascular,1.0,Si,0,NaT,NaN,0,0,0,NaN,NaN,0,52002,0.0
3586,3028,2022-10-05,2022-12-07,None,NaN,Si,0,NaT,NaN,0,0,0,NaN,NaN,0,52002,0.0
3599,3041,2023-04-01,2023-06-18,None,NaN,Si,1,2023-06-18,5.0,1,1,78,22.0,1.0,0,41013,NaN
3600,3041,2023-04-01,2023-06-18,None,NaN,Si,1,2023-06-18,44.0,0,1,78,NaN,NaN,0,41013,NaN


In [10]:
# Keep hemoculture with max days to mortality
tbl_episodios_max = (
    tbl_episodios.groupby(["record_id", "fecha_ingreso"], group_keys=False)
                 .apply(lambda g: g.nlargest(1, "dias_hemocultivo_mortalidad"))
                 .reset_index(drop=True)
)
# print shape
print(tbl_episodios_max.shape)

(3412, 17)


In [11]:
# Code if deaths within 14 days
tbl_episodios_max['fecha_ingreso'] = pd.to_datetime(
    tbl_episodios_max['fecha_ingreso'],
    errors='coerce'
).dt.date

tbl_episodios_max['fecha_mortalidad'] = pd.to_datetime(
    tbl_episodios_max['fecha_mortalidad'],
    errors='coerce'
).dt.date

## Code mortalidad at 14 days too
tbl_episodios_max['mortalidad_14_dias'] = np.where(
    (tbl_episodios_max['fecha_mortalidad'] - tbl_episodios_max['fecha_ingreso']) <= pd.Timedelta(days=14),1,0)

In [163]:
# UCI before hemocultivo
tbl_episodios_max['en_uci_antes_del_hemocultivo'] = tbl_episodios_max['en_uci_antes_del_hemocultivo'].fillna(0)
# Recode IRAs_nosocomial to '0/1'
tbl_episodios_max['IRAs_nosocomial'] = tbl_episodios_max['IRAs_nosocomial'].map({'No': 0, 'Si': 1})
tbl_episodios_max.drop(columns=['codigo_postal'], inplace=True)

In [164]:
tbl_episodios_max['organo_aparato'].value_counts(dropna=False)

organo_aparato
None                                      2103
Infección de la vía urinaria superior      435
Fiebre sin foco                            262
Infeccion de cateter vascular              152
Infeccion vias biliares                    136
Infeccion intraabdominal                   124
Infeccion tracto respiratorio inferior      92
Infeccion de piel y partes blandas          73
Infeccion cardiovascular                    14
Infeccion osteoarticular                     9
Infeccion tracto respiratorio superior       8
Infeccion del SNC                            2
Infeccion genital                            1
Infección dental                             1
Name: count, dtype: int64

In [165]:
# rename organo aparato values
infecname_to_infeccode = {
    'via_urinaria_superior': 'Infección de la vía urinaria superior',
    'fiebre_sin_foco': 'Fiebre sin foco',
    'cateter_vascular': 'Infeccion de cateter vascular',
    'vias_biliares': 'Infeccion vias biliares',
    'intraabdominal': 'Infeccion intraabdominal',
    'tracto_respiratorio_inferior': 'Infeccion tracto respiratorio inferior',
    'piel': 'Infeccion de piel y partes blandas',
    'cardiovascular': 'Infeccion cardiovascular',
    'osteoarticular': 'Infeccion osteoarticular',
    'snc': 'Infeccion del SNC',
    'genital': 'Infeccion genital',
    'etiologia_desconocida': 'etiologia_desconocida'
}


In [166]:
# rename organo_aparato
tbl_episodios_max['organo_aparato'] = tbl_episodios_max['organo_aparato'].fillna('etiologia_desconocida')
tbl_episodios_max['organo_aparato'] = tbl_episodios_max['organo_aparato'].map({v: k for k, v in infecname_to_infeccode.items()})

# One-hot encode the 'organo_aparato' column
tbl_episodios_recoded = pd.get_dummies(tbl_episodios_max, columns=['organo_aparato'], prefix='foco',dtype=int)

# drop foco_controlable column, coded in foco dummies
tbl_episodios_recoded.drop(columns=['foco_controlable'], inplace=True)
tbl_episodios_recoded.head()


,record_id,fecha_ingreso,fecha_alta,IRAs_nosocomial,mortalidad,fecha_mortalidad,dias_hemocultivo_mortalidad,mortalidad_30_dias,uci_por_el_episodio,duracion_UCI,...,foco_etiologia_desconocida,foco_fiebre_sin_foco,foco_genital,foco_intraabdominal,foco_osteoarticular,foco_piel,foco_snc,foco_tracto_respiratorio_inferior,foco_via_urinaria_superior,foco_vias_biliares
0,1,2021-08-24,2021-09-09,1,1,2021-09-09,9.0,1,1,14,...,1,0,0,0,0,0,0,0,0,0
1,2,2023-06-18,2023-07-13,1,0,NaT,NaN,0,1,3,...,1,0,0,0,0,0,0,0,0,0
2,3,2022-02-11,2022-04-06,1,0,NaT,NaN,0,1,24,...,0,0,0,0,0,0,0,0,0,0
3,4,2021-05-15,2021-07-28,1,0,NaT,NaN,0,1,42,...,1,0,0,0,0,0,0,0,0,0
4,5,2021-07-04,2022-01-04,1,0,NaT,NaN,0,1,78,...,0,0,0,0,0,0,0,1,0,0


In [167]:
# evaluate mortality at 30 days time window
print(f"Total mortality: {(tbl_episodios_recoded['mortalidad']==1).sum()}")
print(f"Mortality at 30 days: {(tbl_episodios_recoded['mortalidad_30_dias']==1).sum()}")
print(f"Mortality at 14 days: {(tbl_episodios_recoded['mortalidad_14_dias']==1).sum()}")

Total mortality: 688
Mortality at 30 days: 594
Mortality at 14 days: 294


## **tbl_comorbilidad**
- Clusterizar cancer en clases y pivotar en columnas y codificar **0/1**

In [168]:
tbl_comorbilidades = pd.read_sql_query("SELECT * FROM comorbilidad;", con=conn)
tbl_comorbilidades['fecha_ingreso'] = pd.to_datetime(tbl_comorbilidades['fecha_ingreso'], errors='coerce').dt.date
tbl_comorbilidades.head()

,record_id,fecha_ingreso,infarto,insuficiencia_cardiaca,evp,e_cerebrovascular,demencia,e_pulmonar_cronica,ulcera_peptica,colagenopatia,...,diabetes_sin_lesion_organo_diana,diabetes_con_lesion_organo_diana,inmunosupresion,causa_inmunosupresion,fecha_TOS,TOS,fecha_TPH,TPH,clasificacion_quemadura,gran_quemado
0,1,2021-08-24,0,0,0,0,0,0,0,0,...,0,0,0,None,None,0,None,0,None,0
1,2,2023-06-18,0,0,0,0,0,0,0,0,...,0,0,0,None,None,0,None,0,None,0
2,3,2022-02-11,0,0,0,0,0,0,0,0,...,0,0,0,None,None,0,None,0,None,0
3,4,2021-05-15,0,0,0,0,0,0,0,0,...,0,0,0,None,None,0,None,0,"T20.29XA, T21.22XA, T21.24XA, T24.291A, T22.29...",1
4,5,2021-07-04,0,0,0,0,0,0,0,0,...,0,1,1,58606001,None,0,None,0,None,0


In [169]:
# print shape
print(tbl_comorbilidades.shape)

(3412, 34)


**Classify cancer codes in broad categories (not done)**

- Metadata extracted from https://www.icd10data.com/ICD10CM/Codes/C00-D49

In [170]:
tbl_comorbilidades_recoded = tbl_comorbilidades.copy()

tbl_comorbilidades_recoded['has_cancer'] = np.where(
    tbl_comorbilidades_recoded[
        ['neoplasia_tratamiento_activo',
         'neoplasia_solida_metastasica',
         'neoplasia_solida_no_metastasica']
    ].sum(axis=1) > 0,1,0)

# tbl_comorbilidades_["tipo_cancer_list"] = (
#     tbl_comorbilidades_["tipo_cancer"]
#         .dropna()
#         .apply(lambda x: x.split(","))
# )

# # explode
# tbl_comorbilidades_exploded = (
#     tbl_comorbilidades_
#         .assign(tipo_cancer_list=tbl_comorbilidades_["tipo_cancer_list"])
#         .explode("tipo_cancer_list")
# )
# # recode tipo_cancer
# tbl_comorbilidades_exploded["tipo_cancer_list"] = (
#     tbl_comorbilidades_exploded["tipo_cancer_list"]
#         .astype(str)
#         .str.strip()
#         .str.split(".")
#         .str[0]
# )
# # classify using function defined in utils.py
# tbl_comorbilidades_exploded["cancer_class"] = (
#     tbl_comorbilidades_exploded["tipo_cancer_list"]
#         .map(classify_cancer)
# )

# # create dummies with cancer classes
# dummies = (
#     pd.get_dummies(tbl_comorbilidades_exploded["cancer_class"],
#                    prefix="has_cancer",
#                    dtype=int)
#     .assign(record_id=tbl_comorbilidades_exploded["record_id"])
#     .groupby("record_id")
#     .max()
# )

# # merge w/ tbl_comorbilidades
# tbl_comorbilidades_recoded = tbl_comorbilidades.merge(
#     dummies,
#     on="record_id",
#     how="left"
# )

# # Drop unneded columns
# # child pugh score, coded in hepatopatia_ligera/hepatopatia_moderada_grave
tbl_comorbilidades_recoded.drop(columns=["tipo_hepatopatia","causa_inmunosupresion", 
                                         "fecha_TOS", "fecha_TPH", "clasificacion_quemadura", "puntaje_child_pugh"], inplace=True)

# ncount total comorbidities, exluding cancer types
comorbidity_cols = tbl_comorbilidades_recoded.columns.difference(['record_id', 'fecha_ingreso']+
                                                                 [col for col in tbl_comorbilidades_recoded.columns if 'has_cancer' not in col])
tbl_comorbilidades_recoded['num_comorbilidades'] = tbl_comorbilidades_recoded[comorbidity_cols].sum(axis=1)
tbl_comorbilidades_recoded.head()


,record_id,fecha_ingreso,infarto,insuficiencia_cardiaca,evp,e_cerebrovascular,demencia,e_pulmonar_cronica,ulcera_peptica,colagenopatia,...,hepatopatia_moderada_o_grave,diabetes,diabetes_sin_lesion_organo_diana,diabetes_con_lesion_organo_diana,inmunosupresion,TOS,TPH,gran_quemado,has_cancer,num_comorbilidades
0,1,2021-08-24,0,0,0,0,0,0,0,0,...,NaN,0,0,0,0,0,0,0,1,1
1,2,2023-06-18,0,0,0,0,0,0,0,0,...,NaN,0,0,0,0,0,0,0,1,1
2,3,2022-02-11,0,0,0,0,0,0,0,0,...,NaN,0,0,0,0,0,0,0,1,1
3,4,2021-05-15,0,0,0,0,0,0,0,0,...,NaN,0,0,0,0,0,0,1,0,0
4,5,2021-07-04,0,0,0,0,0,0,0,0,...,NaN,1,0,1,1,0,0,0,0,0


## **tbl_signos**

In [12]:
tbl_signos = dfs['signos_sintomas'].copy()
tbl_signos

,record_id,fecha_ingreso,sepsis,shock_septico,qsofa,somnolencia_estupor_coma,situacion_funcional_basal,indice_de_charlson,escala_karnofsky,barthel_inf_90,...,lesiones_piel,lesiones_mucosas,cefalea,dolores_articulares,temperatura,frec_cardiaca,frecuencia_respiratoria,tension_arterial_sist,tension_arterial_diast,saturacion_pO2
0,1,2021-08-24,1,1,0,normal,"Normal, sin quejas ni evidencia de enfermedad",11,100.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2,2023-06-18,0,0,0,normal,"Normal, sin quejas ni evidencia de enfermedad",4,100.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,3,2022-02-11,0,0,0,normal,None,2,NaN,NaN,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,4,2021-05-15,1,1,0,normal,Necesita ayuda importante y asistencia médica ...,0,50.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,5,2021-07-04,0,0,0,normal,"Normal, sin quejas ni evidencia de enfermedad",2,100.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3592,3045,2023-06-21,0,0,0,normal,"Normal, sin quejas ni evidencia de enfermedad",0,100.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3593,3046,2023-08-02,0,0,0,normal,"Normal, sin quejas ni evidencia de enfermedad",1,100.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3594,3047,2023-08-30,0,0,0,normal,"Normal, sin quejas ni evidencia de enfermedad",0,100.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3595,3048,2023-09-25,0,0,0,None,"Normal, sin quejas ni evidencia de enfermedad",5,100.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
tbl_signos[tbl_signos['record_id'] == 103]

,record_id,fecha_ingreso,sepsis,shock_septico,qsofa,somnolencia_estupor_coma,situacion_funcional_basal,indice_de_charlson,escala_karnofsky,barthel_inf_90,...,lesiones_piel,lesiones_mucosas,cefalea,dolores_articulares,temperatura,frec_cardiaca,frecuencia_respiratoria,tension_arterial_sist,tension_arterial_diast,saturacion_pO2
128,103,2019-07-29,0,0,0,normal,"Normal, sin quejas ni evidencia de enfermedad",2,100.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
129,103,2019-07-29,0,0,2,normal,"Normal, sin quejas ni evidencia de enfermedad",2,100.0,0.0,...,0.0,0.0,0.0,0.0,39.0,100.0,28.0,100.0,NaN,0.95
130,103,2019-07-29,0,0,0,normal,"Normal, sin quejas ni evidencia de enfermedad",2,100.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.96
131,103,2019-07-29,0,0,0,normal,"Normal, sin quejas ni evidencia de enfermedad",2,100.0,0.0,...,0.0,0.0,0.0,0.0,39.5,75.0,NaN,NaN,NaN,NaN
132,103,2019-07-29,0,0,1,normal,"Normal, sin quejas ni evidencia de enfermedad",2,100.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,80.0,45.0,NaN
133,103,2019-07-29,0,0,1,normal,"Normal, sin quejas ni evidencia de enfermedad",2,100.0,0.0,...,0.0,0.0,0.0,0.0,38.2,NaN,NaN,90.0,50.0,0.99


In [172]:
# drop situacion_funcional_basal
tbl_signos.drop(columns=['situacion_funcional_basal'], inplace=True)

# modify 'somnolencia_estupor_coma'
tbl_signos['somnolencia_estupor_coma'] = np.where(
    tbl_signos['somnolencia_estupor_coma'] == 'normal', 0,
    np.where(tbl_signos['somnolencia_estupor_coma'].isna(), np.nan, 1)
)

In [173]:
#print shape
print(tbl_signos.shape)

(3597, 32)


In [174]:
# List of numeric columns (0/1)
num_cols = [c for c in tbl_signos.columns if c not in ["record_id", "fecha_ingreso"]]

# Collapse numeric columns: if any 1 -> 1, all 0 -> 0, all NaN -> NaN
tbl_signos_collapsed = tbl_signos.groupby(["record_id", "fecha_ingreso"], as_index=False).agg(
    {col: lambda x: int(x.dropna().max()) if x.notna().any() else np.nan for col in num_cols}
)

print(tbl_signos_collapsed.shape)


(3412, 32)


Evaluate vital signs abnormalities based

In [175]:
for c in ['saturacion_pO2', 'frec_cardiaca', 'frecuencia_respiratoria', 'temperatura', 'tension_arterial_sist', 'tension_arterial_diast']:
    print(f"Descriptive statistics for {c}:")
    print(tbl_signos_collapsed[c].describe())

Descriptive statistics for saturacion_pO2:
count    473.000000
mean       0.114165
std        0.318348
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000
Name: saturacion_pO2, dtype: float64
Descriptive statistics for frec_cardiaca:
count    486.000000
mean      96.393004
std       20.463190
min       53.000000
25%       80.000000
50%       95.000000
75%      110.000000
max      180.000000
Name: frec_cardiaca, dtype: float64
Descriptive statistics for frecuencia_respiratoria:
count    83.000000
mean     21.036145
std       5.356659
min      12.000000
25%      17.000000
50%      20.000000
75%      24.000000
max      40.000000
Name: frecuencia_respiratoria, dtype: float64
Descriptive statistics for temperatura:
count    292.000000
mean      37.376712
std        0.999247
min       35.000000
25%       37.000000
50%       37.000000
75%       38.000000
max       40.000000
Name: temperatura, dtype: float64
Descriptive statistics for tension_art

Rename and pivot:

#- *foco*

Recode:
- *somnolencia_estupor_coma*
- *indice_de_charlson (low/medium/high)*
- *escala karnofsky (low/medium/high)*
- *hipotermia/hipertermia*
- *hipotension/hipertension*
- *hipoxemia*
- *taquipnea/taquicardia*

> Keep NaN values as they are

In [176]:
# copy 
tbl_signos_recoded = tbl_signos_collapsed.copy()

In [177]:
# renaming and recoding 'foco'
# tbl_signos['foco'] = tbl_signos['foco'].replace({'piel y partes blandas': 'piel', 'cateter venoso' : 'cateter'})
# tbl_signos_recoded = pd.get_dummies(tbl_signos, columns=['foco'], prefix='foco', dtype=int)

# indice de charlson
tbl_signos_recoded['charlson_index_class'] = pd.cut(tbl_signos['indice_de_charlson'],bins=[-1, 3, 6, float('inf')],labels=['low', 'medium', 'high'])
tbl_signos_recoded = pd.get_dummies(tbl_signos_recoded, columns=['charlson_index_class'], prefix='charlson_index', dtype=int, dummy_na=True)

# escala karnofsky
tbl_signos_recoded['karnofsky_class'] = pd.cut(tbl_signos['escala_karnofsky'],bins=[-1, 50, 80, float('inf')],labels=['high', 'medium', 'low'])
tbl_signos_recoded = pd.get_dummies(tbl_signos_recoded, columns=['karnofsky_class'], prefix='karnofsky', dtype=int, dummy_na=True)

# hipo/hipertermia
tbl_signos_recoded['hipertermia'] = np.where(tbl_signos_recoded['temperatura'].isna(), np.nan,np.where(tbl_signos_recoded['temperatura'] >= 38, 1, 0))
tbl_signos_recoded['hipotermia'] = np.where(tbl_signos_recoded['temperatura'].isna(), np.nan,np.where(tbl_signos_recoded['temperatura'] < 36, 1, 0))

# hipo/hipertensión
tbl_signos_recoded['hipotension'] = np.where(
    tbl_signos_recoded['tension_arterial_sist'].isna() | tbl_signos_recoded['tension_arterial_diast'].isna(),np.nan,
    np.where((tbl_signos_recoded['tension_arterial_sist'] <= 90) & (tbl_signos_recoded['tension_arterial_diast'] <= 60),1, 0)
)
tbl_signos_recoded['hipertension'] = np.where(
    tbl_signos_recoded['tension_arterial_sist'].isna() | tbl_signos_recoded['tension_arterial_diast'].isna(), np.nan,
    np.where((tbl_signos_recoded['tension_arterial_sist'] >= 140) &(tbl_signos_recoded['tension_arterial_diast'] >= 90),1, 0))

# taquipnea/taquicardia
tbl_signos_recoded['taquipnea'] = np.where(tbl_signos_recoded['frecuencia_respiratoria'].isna(),np.nan,np.where(tbl_signos_recoded['frecuencia_respiratoria'] > 20, 1, 0))
tbl_signos_recoded['taquicardia'] = np.where(tbl_signos_recoded['frec_cardiaca'].isna(),np.nan,np.where(tbl_signos_recoded['frec_cardiaca'] > 90, 1, 0))

# hipoxemia
tbl_signos_recoded['hipoxemia'] = np.where(tbl_signos_recoded['saturacion_pO2'].isna(),np.nan,np.where(tbl_signos_recoded['saturacion_pO2'] < 0.90, 1, 0))

# drop unneeded columns
tbl_signos_recoded.drop(columns=['indice_de_charlson', 'escala_karnofsky', 'barthel_inf_90', 'temperatura', 'tension_arterial_sist',
                                  'tension_arterial_diast', 'frec_cardiaca', 'saturacion_pO2'], inplace=True)
tbl_signos_recoded.head()

,record_id,fecha_ingreso,sepsis,shock_septico,qsofa,somnolencia_estupor_coma,fiebre,tos,dificultad_respirar,dolor_costal,...,karnofsky_medium,karnofsky_low,karnofsky_nan,hipertermia,hipotermia,hipotension,hipertension,taquipnea,taquicardia,hipoxemia
0,1,2021-08-24,1,1,0,0.0,0.0,0.0,0.0,0.0,...,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,2023-06-18,0,0,0,0.0,0.0,0.0,0.0,0.0,...,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,2022-02-11,0,0,0,0.0,0.0,0.0,0.0,0.0,...,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,2021-05-15,1,1,0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,2021-07-04,0,0,0,0.0,0.0,0.0,0.0,0.0,...,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Missing indicator
- Create *missing_signos* variable with number of missing vital signs, excluding measurements (a lot of missing values there)


In [178]:
missing_sign_cols = ['somnolencia_estupor_coma','fiebre', 'tos', 'dificultad_respirar',
       'dolor_costal', 'disuria', 'polaquiuria', 'tenesmo_vejiga','tenesmo_ano_recto', 'dolor_fosa_renal', 'nauseas', 'vomitos',
       'dolor_abdominal', 'diarrea', 'lesiones_piel', 'lesiones_mucosas','cefalea', 'dolores_articulares']

# Create missing_sign: 1 if all selected columns are NaN, else 0
tbl_signos_recoded['missing_sign'] = (
    tbl_signos_recoded[missing_sign_cols].isna().all(axis=1).astype(int)
)
tbl_signos_recoded[missing_sign_cols] = tbl_signos_recoded[missing_sign_cols].fillna(0).astype(int)
tbl_signos_recoded['missing_sign'].value_counts()


missing_sign
0    2019
1    1393
Name: count, dtype: int64

## **tbl_microorganismo**

In [179]:
tbl_microorganismo = pd.read_sql_query("SELECT * FROM episodio_infeccion;", con=conn)
tbl_microorganismo = tbl_microorganismo.sort_values(by=["record_id","fecha_ingreso", "fecha_cultivo"])
tbl_microorganismo = tbl_microorganismo.drop(columns=['id_cultivo'])
tbl_microorganismo["fecha_cultivo"] = pd.to_datetime(tbl_microorganismo["fecha_cultivo"]).dt.date
tbl_microorganismo["fecha_ingreso"] = pd.to_datetime(tbl_microorganismo["fecha_ingreso"]).dt.date
tbl_microorganismo.head()

,episode_id,record_id,fecha_ingreso,fecha_cultivo,area_hosp,especimen,microorganismo,fenotipo_resistencia
0,1,1,2021-08-24,2021-08-31,UCI,Sangre,Pseudomonas aeruginosa,None
1,2,2,2023-06-18,2023-06-30,General,Sangre,Escherichia coli,None
2,3,3,2022-02-11,2021-04-23,UCI,Secreción bronquial (aspirado),Klebsiella pneumoniae ssp pneumoniae,BLEE
3,4,3,2022-02-11,2021-04-23,UCI,Secreción bronquial (aspirado),Pseudomonas aeruginosa,None
4,5,3,2022-02-11,2021-04-29,UCI,Secreción bronquial (aspirado),Pseudomonas aeruginosa,None


### Temporally define episodes
- Create ***hemocultivo_si_no***
- ***hemocultivo_principal***  --> first hemocultivo since *fecha_ingreso*
-  ***previous_episodes***: before *hemocultivo_principal*


In [180]:
PRE_ADMISSION_DAYS = 2 # some episodes have cultures taken few days before admission

In [181]:
df = tbl_microorganismo.copy()

# 1) Filter valid blood cultures: Sangre and after admission
valid = df.loc[
    (df["especimen"] == "Sangre") &
    (
        df["fecha_cultivo"] >= 
        (df["fecha_ingreso"] - pd.Timedelta(days=PRE_ADMISSION_DAYS))
    )
].copy()

# 2) Get the first blood culture date per admission
first_hemo_dates = (
    valid.groupby(["record_id", "fecha_ingreso"])["fecha_cultivo"]
         .min()
         .reset_index()
         .rename(columns={"fecha_cultivo": "first_hemo_date"})
)

# Merge back to valid rows
valid = valid.merge(first_hemo_dates, on=["record_id", "fecha_ingreso"])

# 3) Mark hemocultivo_principal for all microorganisms in the first blood culture
valid["hemocultivo_principal"] = (
    valid["fecha_cultivo"] == valid["first_hemo_date"]
).astype(int)

# 4) Merge hemocultivo_principal back to full table
df = df.merge(
    valid[["record_id", "fecha_ingreso", "episode_id", "hemocultivo_principal"]],
    on=["record_id", "fecha_ingreso", "episode_id"],
    how="left"
)
df["hemocultivo_principal"] = df["hemocultivo_principal"].fillna(0).astype(int)

# 5) episodio_previo_si_no: before the first hemoculture
# Map first hemo date per admission
first_hemo_map = first_hemo_dates.set_index(["record_id", "fecha_ingreso"])["first_hemo_date"]
df["fecha_principal_hemo"] = df.set_index(["record_id", "fecha_ingreso"]).index.map(first_hemo_map)

df["episodio_previo_si_no"] = np.where(
    df["fecha_principal_hemo"].notna() & (df["fecha_cultivo"] < df["fecha_principal_hemo"]),
    1, 0
)

# 6) Cleanup helper column
df.drop(columns=["fecha_principal_hemo"], inplace=True)

tbl_microorganismo = df.copy()
tbl_microorganismo.head()

,episode_id,record_id,fecha_ingreso,fecha_cultivo,area_hosp,especimen,microorganismo,fenotipo_resistencia,hemocultivo_principal,episodio_previo_si_no
0,1,1,2021-08-24,2021-08-31,UCI,Sangre,Pseudomonas aeruginosa,None,1,0
1,2,2,2023-06-18,2023-06-30,General,Sangre,Escherichia coli,None,1,0
2,3,3,2022-02-11,2021-04-23,UCI,Secreción bronquial (aspirado),Klebsiella pneumoniae ssp pneumoniae,BLEE,0,1
3,4,3,2022-02-11,2021-04-23,UCI,Secreción bronquial (aspirado),Pseudomonas aeruginosa,None,0,1
4,5,3,2022-02-11,2021-04-29,UCI,Secreción bronquial (aspirado),Pseudomonas aeruginosa,None,0,1


In [182]:
tbl_microorganismo[tbl_microorganismo['episodio_previo_si_no']==1].shape

(2018, 10)

### Classify microorganismos into groups
- Bacterial groups:
  -  **E. coli**
  -  **K. pneumoniae**
  -  **P. aeruginosa** 
  -  **S. aureus**
  -  **Enterococcus**
  -  **Enterobacterias** 
  -  **Other**
  
- Has had a BMR?:
  -  **BMR_main_si_no** (1/0)
  -  **BMR_prev_si_no** (1/0)

In [183]:
tbl = tbl_microorganismo.copy()

# standarize and format microorganism names
tbl["microorganismo_recoded"] = (
    tbl["microorganismo"]
    .fillna("")
    .apply(lambda x: "_".join(x.split(" ")[:2]))
)

# CLASSIFY microorganism into your defined classes 
tbl["microorganismo_group"] = tbl["microorganismo_recoded"].map(classify_microorganism)


# ================
#  ONE-HOT ENCODING:
#    MAIN vs PREVIOUS episodes separately
# ================

# A) MAIN EPISODES (episodio_previo_si_no = 0)
dummies_main = (
    pd.get_dummies(
        tbl.loc[tbl["episodio_previo_si_no"] == 0, "microorganismo_group"],
        prefix="microorganismo_main",
        dtype=int
    )
)

# B) PREVIOUS EPISODES (episodio_previo_si_no = 1)
dummies_prev = (
    pd.get_dummies(
        tbl.loc[tbl["episodio_previo_si_no"] == 1, "microorganismo_group"],
        prefix="microorganismo_prev",
        dtype=int
    )
)

# join main and previous dummies
tbl = tbl.join(dummies_main).join(dummies_prev)

# BMR resistance 
tbl["BMR_main_si_no"] = np.where(
    (tbl["episodio_previo_si_no"] == 0) & (tbl["fenotipo_resistencia"].notna()),
    1, 0
)
tbl["BMR_prev_si_no"] = np.where(
    (tbl["episodio_previo_si_no"] == 1) & (tbl["fenotipo_resistencia"].notna()),
    1, 0
)

# drop unnecessary cols and fill nan
tbl_microorganismo_recoded = tbl.drop(columns=["microorganismo", "fenotipo_resistencia"])
tbl_microorganismo_recoded.fillna(0, inplace=True)

tbl_microorganismo_recoded.head()

,episode_id,record_id,fecha_ingreso,fecha_cultivo,area_hosp,especimen,hemocultivo_principal,episodio_previo_si_no,microorganismo_recoded,microorganismo_group,...,microorganismo_main_Pseudomonas_aeruginosa,microorganismo_main_Staphylococcus_aureus,microorganismo_prev_Enterobacteria,microorganismo_prev_Escherichia_coli,microorganismo_prev_Klebsiella_pneumoniae,microorganismo_prev_Other,microorganismo_prev_Pseudomonas_aeruginosa,microorganismo_prev_Staphylococcus_aureus,BMR_main_si_no,BMR_prev_si_no
0,1,1,2021-08-24,2021-08-31,UCI,Sangre,1,0,Pseudomonas_aeruginosa,Pseudomonas_aeruginosa,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
1,2,2,2023-06-18,2023-06-30,General,Sangre,1,0,Escherichia_coli,Escherichia_coli,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
2,3,3,2022-02-11,2021-04-23,UCI,Secreción bronquial (aspirado),0,1,Klebsiella_pneumoniae,Klebsiella_pneumoniae,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,1
3,4,3,2022-02-11,2021-04-23,UCI,Secreción bronquial (aspirado),0,1,Pseudomonas_aeruginosa,Pseudomonas_aeruginosa,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0,0
4,5,3,2022-02-11,2021-04-29,UCI,Secreción bronquial (aspirado),0,1,Pseudomonas_aeruginosa,Pseudomonas_aeruginosa,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0,0


### Construct collapsed *tbl_microorganismo* (one row per record_id)
- Keep main hemocultivo and previous episodes for ML predictive model
- Code 'ncount_microorganismos' (total number of different microorganisms))
- Group 'microorganismos', 'especimen' and 'fecha_cultivo' as lists
- Keep max value for microorganism groups columns (1/0)
- Drop unneeded columns
  

In [184]:
df = tbl_microorganismo_recoded.copy()

# Keep only main or previous episodes for ML model
df = df.loc[
    (df["episodio_previo_si_no"] == 1) |
    (df["hemocultivo_principal"] == 1)
].copy()

micro_main_cols = [c for c in df.columns if c.startswith("microorganismo_main_")]
micro_prev_cols = [c for c in df.columns if c.startswith("microorganismo_prev_")]

# 2. Count microorganisms
micro_count_main = (
    df.groupby(["record_id", "fecha_ingreso"])[micro_main_cols]
      .sum().sum(axis=1)
      .rename("ncount_microorganismos_main")
)
# more than one microorganism = coinfection
coinfection_main = (
    (micro_count_main > 1)
    .astype(int)
    .rename("coinfection_main_si_no")
)

micro_count_prev = (
    df.groupby(["record_id", "fecha_ingreso"])[micro_prev_cols]
      .sum().sum(axis=1)
      .rename("ncount_microorganismos_prev")
)

# collapse fechas, microorganismos, especimen as lists
microorganismos_dict = (
    df.groupby(["record_id", "fecha_ingreso"])["microorganismo_recoded"]
      .apply(list)
      .rename("microorganismos_dict")
)

fechas_dict = (
    df.groupby(["record_id", "fecha_ingreso"])["fecha_cultivo"]
      .apply(list)
      .rename("fechas_cultivo_dict")
)

especimen_dict = (
    df.groupby(["record_id", "fecha_ingreso"])["especimen"]
      .apply(list)
      .rename("especimen_dict")
)

# Collapse one-hot columns (MAX), if 1 in any episode keep it
df_max_main = (
    df.groupby(["record_id", "fecha_ingreso"])[micro_main_cols]
      .max()
)
df_max_prev = (
    df.groupby(["record_id", "fecha_ingreso"])[micro_prev_cols]
      .max()
)

# Keep first row for metadata
df_first = (
    df.sort_values(["record_id", "fecha_ingreso", "fecha_cultivo"])
      .groupby(["record_id", "fecha_ingreso"])
      .first()
)

df_first[micro_main_cols] = df_max_main
df_first[micro_prev_cols] = df_max_prev

# Merge everything
tbl_microorganismo_collapsed = (
    df_first
    .join(micro_count_main)
    .join(coinfection_main)
    .join(micro_count_prev)
    .join(microorganismos_dict)
    .join(fechas_dict)
    .join(especimen_dict)
    .reset_index()
)


# Drop unneeded columns
tbl_microorganismo_collapsed.drop(
    columns=["especimen", "microorganismo_recoded", "fecha_cultivo", "microorganismo_group"],
    inplace=True,
    errors="ignore"
)

tbl_microorganismo_collapsed.head()
 

,record_id,fecha_ingreso,episode_id,area_hosp,hemocultivo_principal,episodio_previo_si_no,microorganismo_main_Enterobacteria,microorganismo_main_Escherichia_coli,microorganismo_main_Klebsiella_pneumoniae,microorganismo_main_Other,...,microorganismo_prev_Pseudomonas_aeruginosa,microorganismo_prev_Staphylococcus_aureus,BMR_main_si_no,BMR_prev_si_no,ncount_microorganismos_main,coinfection_main_si_no,ncount_microorganismos_prev,microorganismos_dict,fechas_cultivo_dict,especimen_dict
0,1,2021-08-24,1,UCI,1,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,0,1.0,0,0.0,[Pseudomonas_aeruginosa],[2021-08-31],[Sangre]
1,2,2023-06-18,2,General,1,0,0.0,1.0,0.0,0.0,...,0.0,0.0,0,0,1.0,0,0.0,[Escherichia_coli],[2023-06-30],[Sangre]
2,3,2022-02-11,3,UCI,0,1,1.0,0.0,0.0,0.0,...,1.0,0.0,0,1,1.0,0,5.0,"[Klebsiella_pneumoniae, Pseudomonas_aeruginosa...","[2021-04-23, 2021-04-23, 2021-04-29, 2022-02-1...","[Secreción bronquial (aspirado), Secreción bro..."
3,4,2021-05-15,10,UCI,0,1,0.0,0.0,0.0,0.0,...,1.0,0.0,0,0,1.0,0,5.0,"[Pseudomonas_aeruginosa, Pseudomonas_aeruginos...","[2021-06-04, 2021-06-10, 2021-06-15, 2021-06-1...","[Quemadura (escara), Quemadura (escara), Quema..."
4,5,2021-07-04,16,General,0,1,0.0,0.0,0.0,0.0,...,1.0,0.0,0,1,1.0,0,5.0,"[Klebsiella_pneumoniae, Pseudomonas_aeruginosa...","[2021-08-28, 2021-08-28, 2021-09-09, 2021-10-0...","[Esputo, Esputo, Esputo, Esputo, Esputo, Sangre]"


## tbl_antibiograma

In [185]:
tbl_antibiograma = pd.read_sql_query("SELECT * FROM antibiograma", con=conn)
tbl_antibiograma.head()

,episode_id,antimicrobiano,cmi,interpretacion
0,1,Cefazolina,>16 mg/L,R
1,1,Trimetoprim,>4 mg/L,R
2,1,Ertapenem,>1 mg/L,R
3,1,Cefuroxima,>8 mg/L,R
4,1,Trimetroprim/sulfametoxazol,>4/76 mg/L,R


In [186]:
tbl_antibiograma['antimicrobiano'].value_counts()

antimicrobiano
Levofloxacino                  6346
Tobramicina                    6302
Ciprofloxacino                 6300
Amikacina                      6287
Trimetroprim/sulfametoxazol    6077
                               ... 
Ceftazidima/clavulanico           6
Cefotaxima/clavulanico            5
Colistina Microdilución           1
Amoxicilina                       1
Meropenem (E-test)                1
Name: count, Length: 66, dtype: int64

In [187]:
antimicrobial_family_map = {

    # PENICILLINS
    "Penicilina": "Penicillin",
    "Ampicilina": "Penicillin",
    "Amoxicilina": "Penicillin",
    "Oxacilina": "Penicillin",
    "Ticarcilina": "Penicillin",
    "Piperacilina": "Penicillin",
    "Mecillinam": "Penicillin",
    "Temocilina (E-test)": "Penicillin",
    "Amoxicilina/clavulánico": "Penicillin",
    "Ampicilina/sulbactam": "Penicillin",
    "Piperacilina/tazobactam": "Penicillin",
    "Ticarcilina/clavulanico": "Penicillin",

    # CEPHALOSPORINS
    "Cefalexina": "Cefalosporin",
    "Cefalotina": "Cefalosporin",
    "Cefazolina": "Cefalosporin",
    "Cefuroxima": "Cefalosporin",
    "Cefoxitina": "Cefalosporin",
    "Cefotaxima": "Cefalosporin",
    "Ceftazidima": "Cefalosporin",
    "Cefixime": "Cefalosporin",
    "Cefepime": "Cefalosporin",
    "Ceftarolina": "Cefalosporin",
    "Cefiderocol": "Cefalosporin",
    "Ceftazidima/avibactam": "Cefalosporin",
    "Ceftolozano/tazobactam": "Cefalosporin",
    "Cefotaxima/clavulanico": "Cefalosporin",
    "Ceftazidima/clavulanico": "Cefalosporin",

    # CARBAPENEMS
    "Imipenem": "Carbapenem",
    "Meropenem": "Carbapenem",
    "Meropenem (E-test)": "Carbapenem",
    "Ertapenem": "Carbapenem",

    # QUINOLONES
    "Ciprofloxacino": "Quinolone",
    "Levofloxacino": "Quinolone",
    "Moxifloxacino": "Quinolone",
    "Norfloxacino": "Quinolone",
    "Ofloxacino": "Quinolone",
    "Ácido nalidíxico": "Quinolone",

    # AMINOGLYCOSIDES
    "Gentamicina": "Aminoglycoside",
    "Amikacina": "Aminoglycoside",
    "Tobramicina": "Aminoglycoside",
    "Kanamicina": "Aminoglycoside",
    "Netilmicina": "Aminoglycoside",

    # TETRACYCLINES
    "Tetraciclina": "Tetracycline",
    "Doxiciclina": "Tetracycline",
    "Minociclina": "Tetracycline",
    "Tigeciclina": "Tetracycline",

    # TMP-SMX
    "Trimetoprim": "TMP_SMX",
    "Trimetoprim/sulfametoxazol": "TMP_SMX",
    "Sulfametoxazol/trimetoprim": "TMP_SMX",
    "Trimetroprim/sulfametoxazol": "TMP_SMX",

    # AZTREONAM
    "Aztreonam": "Monobactam",

    # ANTI-GRAM+ LAST LINE
    "Vancomicina": "Glyco_Lipo_Oxa",
    "Teicoplanina": "Glyco_Lipo_Oxa",
    "Daptomicina": "Glyco_Lipo_Oxa",
    "Linezolid": "Glyco_Lipo_Oxa",
    "Synercid": "Glyco_Lipo_Oxa",
    "Pristanamicina": "Glyco_Lipo_Oxa",

    # POLYMYXINS
    "Colistina": "Polymyxin",
    "Colistina Microdilución": "Polymyxin",

    # URINARY / NICHE
    "Fosfomicina": "Fosfomycin_Nitro",
    "Nitrofurantoina": "Fosfomycin_Nitro",

    # REMAINING OTHER
    "Cloranfenicol": "Other",
    "Rifampicina": "Other",
    "Clindamicina": "Other",
    "Eritromicina": "Other",
    "Azitromicina": "Other",
    "Mupirocina": "Other",
    "Ácido fusídico": "Other"

}


In [188]:
# map antimicrobiano to antimicrobiano family
tbl_antibiograma['antimicrobiano_family'] = tbl_antibiograma['antimicrobiano'].map(antimicrobial_family_map)

In [189]:
# group antimicrobials by episode
tbl_antibiograma_grouped = pd.DataFrame({
    'antimicrobiano_list': tbl_antibiograma.groupby('episode_id')['antimicrobiano'].apply(list),
    'cmi_list': tbl_antibiograma.groupby('episode_id')['cmi'].apply(list),
    'interpretacion_list': tbl_antibiograma.groupby('episode_id')['interpretacion'].apply(list),
    'antimicrobiano_family_list': tbl_antibiograma.groupby('episode_id')['antimicrobiano_family'].apply(list)
}).reset_index()
tbl_antibiograma_grouped.head()

,episode_id,antimicrobiano_list,cmi_list,interpretacion_list,antimicrobiano_family_list
0,1,"[Cefazolina, Trimetoprim, Ertapenem, Cefuroxim...","[>16 mg/L, >4 mg/L, >1 mg/L, >8 mg/L, >4/76 mg...","[R, R, R, R, R, R, R, R, I, I, I, I, I, I, I, ...","[Cefalosporin, TMP_SMX, Carbapenem, Cefalospor..."
1,2,"[Trimetroprim/sulfametoxazol, Cefuroxima, Cipr...","[>4/76 mg/L, <=8 mg/L, 0.5 mg/L, <=0.5 mg/L, <...","[R, I, I, S, S, S, S, S, S, S, S, S, S, S, S, ...","[TMP_SMX, Cefalosporin, Quinolone, Quinolone, ..."
2,3,"[Trimetroprim/sulfametoxazol, Ampicilina, Tobr...","[>4/76 mg/L, >8 mg/L, >4 mg/L, >1 mg/L, >1 mg/...","[R, R, R, R, R, R, R, R, R, R, R, R, R, R, R, ...","[TMP_SMX, Penicillin, Aminoglycoside, Quinolon..."
3,4,"[Trimetoprim, Ampicilina, Ertapenem, Cefotaxim...","[>4 mg/L, >8 mg/L, >1 mg/L, >32 mg/L, >1 mg/L,...","[R, R, R, R, R, R, R, R, R, R, R, I, I, I, I, ...","[TMP_SMX, Penicillin, Carbapenem, Cefalosporin..."
4,5,"[Trimetoprim, Gentamicina, Ticarcilina, Ampici...","[>4 mg/L, >4 mg/L, >16 mg/L, >8 mg/L, >16 mg/L...","[R, R, R, R, R, R, R, R, R, R, R, R, R, R, R, ...","[TMP_SMX, Aminoglycoside, Penicillin, Penicill..."


## tbl_tto_antimicrobiano

In [190]:
tbl_tto = pd.read_sql_query("SELECT * FROM tto_antimicrobiano", con=conn)
tbl_tto.head()

,episode_id,numero_tratamiento,tipo_tratamiento,antimicrobiano,atc,fecha_inicio,fecha_fin,dias_tratamiento,dosis,intervalo,via_administracion,tratamiento_apropiado
0,1,1,empirico,VANCOMICINA,J01XA,2021-08-31,None,NaN,1,c/12h UCI (9-21h),PERF IV INTER,1.0
1,1,1,dirigido,ERITROMICINA,J01FA,2021-09-06,2021-09-08,2.0,250,c/8h (08-16-00h)-,PERF IV INTER,1.0
2,1,2,dirigido,MEROPENEM,J01DH,2021-09-06,None,NaN,1000,c/8h (08-16-00h)-,PERF IV INTER,1.0
3,1,3,dirigido,CIPROFLOXACINO,J01MA,2021-09-07,2021-09-08,1.0,200,c/12h (08-20h),PERF IV INTER,1.0
4,1,4,dirigido,LINEZOLID,J01X,2021-09-07,None,NaN,600,c/12h (12h-00h) UCIP,PERF IV INTER,1.0


In [191]:
# map antimicrobiano to antimicrobiano family
tbl_tto['antimicrobiano_tto_family'] = tbl_tto['antimicrobiano'].str.capitalize().map(antimicrobial_family_map)

## **Merge encoded tables**

In [192]:
# merge all processed tables for modeling
df_merged = pd.merge(tbl_pacientes, tbl_episodios_recoded, on="record_id", how="left")

# Compute edad
fecha_ingreso_dt = pd.to_datetime(df_merged["fecha_ingreso"], errors="coerce")
fecha_nac_dt = pd.to_datetime(df_merged["fecha_nacimiento"], errors="coerce")
df_merged["edad"] = ((fecha_ingreso_dt - fecha_nac_dt).dt.days // 365)
df_merged.drop(columns=["fecha_nacimiento"], inplace=True)

# keep merging
df_merged = pd.merge(df_merged, tbl_comorbilidades_recoded, on=["record_id","fecha_ingreso"], how="left")
df_merged = pd.merge(df_merged, tbl_microorganismo_collapsed, on=["record_id","fecha_ingreso"], how="left")
df_merged = pd.merge(df_merged, tbl_signos_recoded, on=["record_id","fecha_ingreso"], how="left")
df_merged = pd.merge(df_merged, tbl_factores_bmr, on=["record_id","fecha_ingreso"], how="left")
df_merged = pd.merge(df_merged, tbl_antibiograma_grouped, on="episode_id", how="left")
df_merged.head()   

,record_id,sexo,fecha_ingreso,fecha_alta,IRAs_nosocomial,mortalidad,fecha_mortalidad,dias_hemocultivo_mortalidad,mortalidad_30_dias,uci_por_el_episodio,...,cateter_venoso,sonda_urinaria,sonda_nasogastrica,derivacion_ventriculoper,valvula_prot_cardiaca,portador_otros_disposit,antimicrobiano_list,cmi_list,interpretacion_list,antimicrobiano_family_list
0,1,0,2021-08-24,2021-09-09,1,1,2021-09-09,9.0,1,1,...,0,0,1,0,0,0,"[Cefazolina, Trimetoprim, Ertapenem, Cefuroxim...","[>16 mg/L, >4 mg/L, >1 mg/L, >8 mg/L, >4/76 mg...","[R, R, R, R, R, R, R, R, I, I, I, I, I, I, I, ...","[Cefalosporin, TMP_SMX, Carbapenem, Cefalospor..."
1,2,1,2023-06-18,2023-07-13,1,0,NaT,NaN,0,1,...,0,0,1,0,0,0,"[Trimetroprim/sulfametoxazol, Cefuroxima, Cipr...","[>4/76 mg/L, <=8 mg/L, 0.5 mg/L, <=0.5 mg/L, <...","[R, I, I, S, S, S, S, S, S, S, S, S, S, S, S, ...","[TMP_SMX, Cefalosporin, Quinolone, Quinolone, ..."
2,3,0,2022-02-11,2022-04-06,1,0,NaT,NaN,0,1,...,0,0,1,0,0,0,"[Trimetroprim/sulfametoxazol, Ampicilina, Tobr...","[>4/76 mg/L, >8 mg/L, >4 mg/L, >1 mg/L, >1 mg/...","[R, R, R, R, R, R, R, R, R, R, R, R, R, R, R, ...","[TMP_SMX, Penicillin, Aminoglycoside, Quinolon..."
3,4,0,2021-05-15,2021-07-28,1,0,NaT,NaN,0,1,...,0,0,1,0,0,0,"[Trimetoprim, Levofloxacino, Ertapenem, Cefazo...","[>4 mg/L, >1 mg/L, >1 mg/L, >16 mg/L, >32 mg/L...","[R, R, R, R, R, R, R, R, R, R, I, I, I, I, I, ...","[TMP_SMX, Quinolone, Carbapenem, Cefalosporin,..."
4,5,1,2021-07-04,2022-01-04,1,0,NaT,NaN,0,1,...,0,0,0,0,0,0,"[Aztreonam, Ciprofloxacino, Ceftazidima, Cefur...","[<=1 mg/L, >1 mg/L, <=1 mg/L, >8 mg/L, 8 mg/L,...","[R, R, R, R, R, R, R, R, R, R, R, R, R, R, R, ...","[Monobactam, Quinolone, Cefalosporin, Cefalosp..."


In [193]:
print(tbl_pacientes.shape)
print(tbl_episodios_recoded.shape)
print(tbl_comorbilidades_recoded.shape)
print(tbl_microorganismo_collapsed.shape)
print(tbl_signos_recoded.shape)
print(tbl_factores_bmr.shape)
#print(tbl_antibiograma_grouped.shape)
print(df_merged.shape)

(3049, 3)
(3412, 27)
(3412, 30)
(3407, 26)
(3412, 40)
(3413, 16)
(3413, 137)


### Mismatches between records and tbl_microorganismo_collapsed

In [194]:
df_merged[df_merged['episodio_previo_si_no']==1].shape

(1024, 137)

In [195]:
# check if mismatches between records and tbl_microorganismo_collapsed
df_merged[df_merged['hemocultivo_principal'].isna()][['record_id']]

,record_id
62,60
477,427
1500,1341
1943,1741
2651,2356


In [196]:
# drop this episodes until solve the mismatch issue
mismatches_record = [60,427,1341,1741,2356]
df_merged = df_merged[~df_merged['record_id'].isin(mismatches_record)]

### NaN values - % of missingness

In [197]:
df_ = df_merged[df_merged["area_hosp"]=="Urgencias"]

for col in df_.columns:
    print(f"{col}: {np.divide(df_[col].isna().sum(), len(df_)) * 100:.2f}% NaN values")

record_id: 0.00% NaN values
sexo: 0.00% NaN values
fecha_ingreso: 0.00% NaN values
fecha_alta: 0.00% NaN values
IRAs_nosocomial: 0.00% NaN values
mortalidad: 0.00% NaN values
fecha_mortalidad: 83.45% NaN values
dias_hemocultivo_mortalidad: 83.45% NaN values
mortalidad_30_dias: 0.00% NaN values
uci_por_el_episodio: 0.00% NaN values
duracion_UCI: 0.00% NaN values
dias_hemocultivo_ingresoUCI: 98.06% NaN values
en_uci_antes_del_hemocultivo: 0.00% NaN values
mujer_gestante: 0.00% NaN values
paciente_residencia: 8.90% NaN values
mortalidad_14_dias: 0.00% NaN values
foco_cardiovascular: 0.00% NaN values
foco_cateter_vascular: 0.00% NaN values
foco_etiologia_desconocida: 0.00% NaN values
foco_fiebre_sin_foco: 0.00% NaN values
foco_genital: 0.00% NaN values
foco_intraabdominal: 0.00% NaN values
foco_osteoarticular: 0.00% NaN values
foco_piel: 0.00% NaN values
foco_snc: 0.00% NaN values
foco_tracto_respiratorio_inferior: 0.00% NaN values
foco_via_urinaria_superior: 0.00% NaN values
foco_vias_bil

In [198]:
NA_THRESHOLD = 20

In [199]:
na_tbl =df_merged.isna().sum().sort_values(ascending=False)
imputable_vars = []
drop_vars = []

print("Imputable Vars:")
for col in na_tbl[na_tbl > 0].index:
    pct = na_tbl[col] / len(df_merged) * 100
    if pct > 0 and pct <= NA_THRESHOLD:
        imputable_vars.append(col)
        print(f"{col}: {na_tbl[col]} NaN --> {pct:.2f}%")

print("\nDROP VARIABLES:")
for col in na_tbl[na_tbl > 0].index:
    pct = na_tbl[col] / len(df_merged) * 100
    if pct > NA_THRESHOLD:
        drop_vars.append(col)
        print(f"{col}: {na_tbl[col]} NaN --> {pct:.2f}%")

Imputable Vars:
antimicrobiano_family_list: 541 NaN --> 15.90%
cmi_list: 541 NaN --> 15.90%
interpretacion_list: 541 NaN --> 15.90%
antimicrobiano_list: 541 NaN --> 15.90%
paciente_residencia: 368 NaN --> 10.82%
cirugia_previa_con_implante: 368 NaN --> 10.82%
asistencia_sanitaria_prev: 368 NaN --> 10.82%
cirugia_previa_sin_implante: 368 NaN --> 10.82%

DROP VARIABLES:
hepatopatia_ligera: 3356 NaN --> 98.65%
hepatopatia_moderada_o_grave: 3356 NaN --> 98.65%
taquipnea: 3318 NaN --> 97.53%
frecuencia_respiratoria: 3318 NaN --> 97.53%
hipotermia: 3111 NaN --> 91.45%
hipertermia: 3111 NaN --> 91.45%
dias_hemocultivo_ingresoUCI: 3036 NaN --> 89.24%
hipoxemia: 2929 NaN --> 86.10%
taquicardia: 2917 NaN --> 85.74%
hipotension: 2860 NaN --> 84.07%
hipertension: 2860 NaN --> 84.07%
dias_hemocultivo_mortalidad: 2715 NaN --> 79.81%
fecha_mortalidad: 2715 NaN --> 79.81%
tipo_cancer: 2137 NaN --> 62.82%


- Drop columns with more than 20% missingness

In [200]:
# drop variables with too many missing values
df_merged = df_merged.drop(columns=drop_vars)
print(f"Dropped {len(drop_vars)} variables with more than {NA_THRESHOLD}% missing values.")
print(f"Variables dropped: {drop_vars}")

Dropped 14 variables with more than 20% missing values.
Variables dropped: ['hepatopatia_ligera', 'hepatopatia_moderada_o_grave', 'taquipnea', 'frecuencia_respiratoria', 'hipotermia', 'hipertermia', 'dias_hemocultivo_ingresoUCI', 'hipoxemia', 'taquicardia', 'hipotension', 'hipertension', 'dias_hemocultivo_mortalidad', 'fecha_mortalidad', 'tipo_cancer']


- Create missigness indicators for imputable variables (asistencia_previa - 488 episodes)

In [201]:
## Create indicator variables for imputable columns
imputable_vars_ = [var for var in imputable_vars if 'list' not in var]
df_merged['ingreso_prev_indicator'] = np.where(df_merged[imputable_vars_].sum(axis=1), 0, 1)
df_merged[imputable_vars] = df_merged[imputable_vars].fillna(0)
print(f"Created indicator variable for imputable vars: ingreso_prev_indicator")

Created indicator variable for imputable vars: ingreso_prev_indicator


In [202]:
# df dimensionality after dropping columns
print(f"Dataframe shape after dropping columns: {df_merged.shape}")

Dataframe shape after dropping columns: (3402, 124)


### Save preprocessed df

In [203]:
### Save merged dataframe
df_merged.to_csv(f"{DATA_DIR}/preprocessed_db.csv", index=False)

In [204]:
## Generate descriptive statistics
# profile = ProfileReport(df_merged, title="Bacthecom Data Report", explorative=True)
# profile.to_notebook_iframe()
# profile.to_file(os.path.join(f"{RESULTS_DIR}", "bacthecom_report.html"))